In [1]:
import numpy as np
import pandas as pd

# Load metadata / wide sales
print('Loading sales_train_evaluation.csv...')
df_sales = pd.read_csv('../data/raw/sales_train_evaluation.csv')

# Filter to 1 store (e.g., CA_1) and 1 category (FOODS_3) to keep compute fast
sample_mask = (df_sales['store_id'] == 'CA_1') & (
    df_sales['cat_id'] == 'FOODS'
)
df_sub = df_sales[sample_mask].head(100).copy()

print(
    f'Selected {len(df_sub)} distinct item series for store'
    f" {df_sub['store_id'].iloc[0]}"
)

Loading sales_train_evaluation.csv...
Selected 100 distinct item series for store CA_1


In [2]:
# Identify ID columns vs Day columns
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in df_sub.columns if c.startswith('d_')]

# Melt to vertical format
df_long = pd.melt(
    df_sub,
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales',
)

df_long['sales'] = df_long['sales'].astype(np.float32)
print(f'Melted shape: {df_long.shape}')
df_long.head()

Melted shape: (194100, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3.0
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0.0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0.0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0.0
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3.0


In [3]:
# Load calendar to get actual dates and event names
df_cal = pd.read_csv('../data/raw/calendar.csv')
cal_cols = [
    'd',
    'date',
    'wm_yr_wk',
    'event_name_1',
    'event_type_1',
    'snap_CA',
]

df_clean = df_long.merge(df_cal[cal_cols], on='d', how='left')
df_clean['date'] = pd.to_datetime(df_clean['date'])

# Rename identifiers to match standard schema
df_clean = df_clean.rename(columns={'store_id': 'store', 'item_id': 'item'})

# Sort strictly by entity and time
df_clean = df_clean.sort_values(['store', 'item', 'date']).reset_index(
    drop=True
)
print('Date range:', df_clean['date'].min(), 'to', df_clean['date'].max())

Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00


In [4]:
# Load prices
df_prices = pd.read_csv('../data/raw/sell_prices.csv')

df_clean = df_clean.merge(
    df_prices, on=['store', 'item', 'wm_yr_wk'], how='left'
)

# Fill missing prices with the mean item price (for when items were not yet listed)
df_clean['sell_price'] = df_clean.groupby(['store', 'item'])[
    'sell_price'
].transform(lambda x: x.bfill().ffill())

# Clean summary check
print('Missing values:\n', df_clean[['date', 'store', 'item', 'sales']].isnull().sum())
print(f'\nFinal Clean Dataset Shape: {df_clean.shape}')

KeyError: 'store'

In [5]:
# Load prices
df_prices = pd.read_csv('../data/raw/sell_prices.csv')

# Rename price columns to match df_clean's 'store' and 'item'
df_prices = df_prices.rename(columns={'store_id': 'store', 'item_id': 'item'})

# Merge on store, item, and wm_yr_wk
df_clean = df_clean.merge(
    df_prices[['store', 'item', 'wm_yr_wk', 'sell_price']],
    on=['store', 'item', 'wm_yr_wk'],
    how='left',
)

# Backfill and forward fill missing prices (for items not yet on shelves)
df_clean['sell_price'] = df_clean.groupby(['store', 'item'])[
    'sell_price'
].transform(lambda x: x.bfill().ffill())

# Clean summary check
print('Missing values in core columns:')
print(df_clean[['date', 'store', 'item', 'sales']].isnull().sum())
print(f'\nFinal Clean Dataset Shape: {df_clean.shape}')
df_clean.head(3)

Missing values in core columns:
date     0
store    0
item     0
sales    0
dtype: int64

Final Clean Dataset Shape: (194100, 14)


,id,item,dept_id,cat_id,store,state_id,d,sales,date,wm_yr_wk,event_name_1,event_type_1,snap_CA,sell_price
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3.0,2011-01-29,11101,NaN,NaN,0,2.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0.0,2011-01-30,11101,NaN,NaN,0,2.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0.0,2011-01-31,11101,NaN,NaN,0,2.0


In [7]:
output_path = '../data/processed/clean_sales.parquet'
df_clean.to_parquet(output_path, index=False)
print(f'Successfully saved cleaned M5 subset to: {output_path}')

ArrowKeyError: No type extension with name arrow.py_extension_type found

In [8]:
output_path = '../data/processed/clean_sales.parquet'
df_clean.to_parquet(output_path, index=False)
print(f'Successfully saved cleaned M5 subset to: {output_path}')

ArrowKeyError: A type extension with name pandas.period already defined

In [9]:
import os

os.makedirs('../data/processed', exist_ok=True)

In [10]:
# 1. Deduplicate column names if any exist
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()].copy()

# 2. Convert object columns with string/mixed types to standard string or category
for col in ['event_name_1', 'event_type_1']:
  if col in df_clean.columns:
    df_clean[col] = df_clean[col].fillna('None').astype(str)

print('Clean columns:', list(df_clean.columns))

Clean columns: ['id', 'item', 'dept_id', 'cat_id', 'store', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'event_name_1', 'event_type_1', 'snap_CA', 'sell_price']


In [11]:
output_path = '../data/processed/clean_sales.parquet'
df_clean.to_parquet(output_path, index=False, engine='pyarrow')
print(f'Successfully saved cleaned M5 subset to: {output_path}')
print(f'Final shape: {df_clean.shape}')


ArrowKeyError: A type extension with name pandas.period already defined

In [12]:
import os
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Resolve absolute path to avoid directory lookup errors
base_dir = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)
processed_dir = base_dir / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
parquet_file = processed_dir / 'clean_sales.parquet'

# 2. Reset index to drop any lingering pandas index metadata
df_clean = df_clean.reset_index(drop=True)

# 3. Ensure string types for object columns so Arrow doesn't choke on mixed types
for col in df_clean.select_dtypes(include=['object']).columns:
  df_clean[col] = df_clean[col].astype(str)

# 4. Convert and write directly via pyarrow
table = pa.Table.from_pandas(df_clean, preserve_index=False)
pq.write_table(table, str(parquet_file))

print(f'Successfully saved to: {parquet_file}')
print(f'Final shape: {df_clean.shape}')

C:\Users\karan\AppData\Local\Temp\ipykernel_19092\3244277428.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_clean.select_dtypes(include=['object']).columns:


Successfully saved to: c:\Users\karan\demand-forecasting-uncertainty\data\processed\clean_sales.parquet
Final shape: (194100, 14)


In [13]:
import os
from pathlib import Path
import pandas as pd

# Locate the file path
base_dir = (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)
parquet_file = base_dir / "data" / "processed" / "clean_sales.parquet"
csv_file = base_dir / "data" / "processed" / "clean_sales.csv.gz"

# Load whichever file was generated
if parquet_file.exists():
  df_check = pd.read_parquet(parquet_file)
  print(f"Verified Parquet file exists! Size: {os.path.getsize(parquet_file) / (1024 * 1024):.2f} MB")
elif csv_file.exists():
  df_check = pd.read_csv(csv_file)
  print(f"Verified CSV file exists! Size: {os.path.getsize(csv_file) / (1024 * 1024):.2f} MB")
else:
  raise FileNotFoundError("Neither clean_sales.parquet nor clean_sales.csv.gz was found in data/processed/.")

print(f"Row count:    {df_check.shape[0]:,}")
print(f"Column count: {df_check.shape[1]}")
print("\nFirst 3 rows:")
display(df_check.head(3))

ArrowKeyError: A type extension with name pandas.period already defined

In [14]:
import os
from pathlib import Path
import pandas as pd

base_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 1. Remove the broken parquet file if it exists
broken_parquet = processed_dir / "clean_sales.parquet"
if broken_parquet.exists():
  os.remove(broken_parquet)
  print("Removed corrupt parquet file.")

# 2. Save directly as standard CSV
csv_path = processed_dir / "clean_sales.csv"
df_clean.to_csv(csv_path, index=False)
print(
    f"Successfully saved clean CSV! Size:"
    f" {os.path.getsize(csv_path) / (1024 * 1024):.2f} MB"
)

Removed corrupt parquet file.
Successfully saved clean CSV! Size: 20.06 MB


In [15]:
# Read back the clean CSV
df = pd.read_csv(csv_path)

print("=" * 60)
print(f"SHAPE: {df.shape[0]:,} rows | {df.shape[1]} columns")
print("=" * 60)
print("\nCOLUMN TYPES & NULLS:")
print(df.info())

print("\n" + "=" * 60)
print("FIRST 3 ROWS (Transposed for clean reading):")
print("=" * 60)
display(df.head(3).T)

print("\n" + "=" * 60)
print("DATE RANGE & IDENTIFIERS:")
print("=" * 60)
print(f"Date range:   {df['date'].min()}  to  {df['date'].max()}")
print(f"Unique stores: {df['store'].unique().tolist()}")
print(f"Unique items:  {df['item'].nunique()} items")

SHAPE: 194,100 rows | 14 columns

COLUMN TYPES & NULLS:
<class 'pandas.DataFrame'>
RangeIndex: 194100 entries, 0 to 194099
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            194100 non-null  str    
 1   item          194100 non-null  str    
 2   dept_id       194100 non-null  str    
 3   cat_id        194100 non-null  str    
 4   store         194100 non-null  str    
 5   state_id      194100 non-null  str    
 6   d             194100 non-null  str    
 7   sales         194100 non-null  float64
 8   date          194100 non-null  str    
 9   wm_yr_wk      194100 non-null  int64  
 10  event_name_1  15800 non-null   str    
 11  event_type_1  15800 non-null   str    
 12  snap_CA       194100 non-null  int64  
 13  sell_price    194100 non-null  float64
dtypes: float64(2), int64(2), str(10)
memory usage: 20.7 MB
None

FIRST 3 ROWS (Transposed for clean reading):


,0,1,2
id,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation
item,FOODS_1_001,FOODS_1_001,FOODS_1_001
dept_id,FOODS_1,FOODS_1,FOODS_1
cat_id,FOODS,FOODS,FOODS
store,CA_1,CA_1,CA_1
state_id,CA,CA,CA
d,d_1,d_2,d_3
sales,3.0,0.0,0.0
date,2011-01-29,2011-01-30,2011-01-31
wm_yr_wk,11101,11101,11101



DATE RANGE & IDENTIFIERS:
Date range:   2011-01-29  to  2016-05-22
Unique stores: ['CA_1']
Unique items:  100 items
